# Aula 3 — Modelo Preditivo

Treino de um classificador (Casa/Empate/Fora) com split temporal, comparação com baseline e análise de erros.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px

from src.data import preparar_dados
from src.models.treinar import treinar, split_temporal
from src.data.paths import DATASET_MODELAGEM

preparar_dados.executar()
ds = pd.read_csv(DATASET_MODELAGEM)
print('dataset:', ds.shape)

## Split temporal
Treino nas rodadas antigas, teste nas recentes. NUNCA embaralhar (o futuro vazaria para o passado).

In [ ]:
treino, teste, corte = split_temporal(ds)
print(f'treino={len(treino)} (rodadas < {corte})  |  teste={len(teste)} (rodadas >= {corte})')

## Treino e métricas

In [ ]:
pac = treinar(ds)
print('baseline (mandante sempre vence):', round(pac['baseline_acuracia'], 3))
for nome, m in pac['metricas'].items():
    print(f"{nome:22s} acuracia={m['acuracia']:.3f}  log_loss={m['log_loss']:.3f}")
print('escolhido:', pac['nome_modelo'])

## Matriz de confusão
Onde o modelo erra? (empates costumam ser os mais difíceis)

In [ ]:
classes = ['Casa', 'Empate', 'Fora']
px.imshow(pac['matriz_confusao'], x=classes, y=classes, text_auto=True,
          labels={'x': 'Previsto', 'y': 'Real'}, color_continuous_scale='Blues')

## Importância das features
Quais variáveis o modelo mais usou? Fecha o ciclo com a EDA da Fase 2.

In [ ]:
imp = pac['importancias'].head(10).sort_values()
px.bar(x=imp.values, y=imp.index, orientation='h', labels={'x': 'Importância', 'y': ''})

## Reflexão
O modelo é bom o suficiente para apostar? Por que não? (Poucos dados, empates difíceis, futebol é imprevisível de propósito.)